In [1]:


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import random
import matplotlib.pyplot as plt
import math     

torch.__version__
torchvision.__version__
import brevitas
torch.cuda.is_available()    
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {device}")   
torch.manual_seed(42)
torch.cuda.manual_seed(42)
random.seed(42)       

print(F)


Using Device: cpu
<module 'torch.nn.functional' from '/home/fede/miniconda3/envs/ptorch/lib/python3.9/site-packages/torch/nn/functional.py'>


No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda-11.8'


In [2]:
from brevitas.core.quant import QuantType
from brevitas.core.restrict_val import RestrictValueType
from brevitas.core.scaling import ScalingImplType

W = int(8)   # total bits
I = int(1)   # integer bits including sign
Fr = W - I

In [3]:
# This is a simple implementation of a quantized Vision Transformer (ViT) using Brevitas.
# It demonstrates how to use Brevitas for quantizing the weights and activations of a ViT model.
# First, we need to install the required libraries.
from brevitas.nn import QuantConv2d, QuantLinear, QuantReLU 
from brevitas.quant import Int8ActPerTensorFloat, Int8WeightPerTensorFloat
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

In [4]:
# The Vision Transformer is composed of several components:
# 1. Patch Embedding: Divides the input image into patches and embeds them into a lower-dimensional space.
# 2. Multi-Head Self-Attention: Computes attention scores for each patch and aggregates information from all patches.
# 3. Feed-Forward Neural Network: Applies a feed-forward network to each patch independently.
# 4. Layer Normalization and Residual Connections: Normalizes the output and adds skip connections.

In [5]:
config = {
    "batch_size":32,
    "patch_size": 4,
    "mlp_size": 256,
    "embed_dim": 128,
    "num_heads": 8,
    "dropout": 0.1,
    "image_size": 28,
    "num_classes": 10,
    "num_channels": 1,
    "qkv_bias": True,
    "depth": 1,
    "epochs": 5
}

In [7]:
# For test/val
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])

# For training (with augmentations)
transform_train = transforms.Compose([
    #transforms.RandomCrop(28, padding=4),
    transforms.RandomHorizontalFlip(),
    #transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5), (0.5))
])
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform_train
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform_test
)

train_loader = DataLoader(train_dataset, batch_size=config["batch_size"])
test_loader = DataLoader(test_dataset, batch_size=config["batch_size"])



def train(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

def test(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total


     

In [8]:
from brevitas.quant import Int8BiasPerTensorFixedPointInternalScaling, Int8WeightPerTensorFixedPoint
from brevitas.quant import Int8ActPerTensorFixedPoint
from brevitas.nn import QuantIdentity


class QuantPatchEmbeddings(nn.Module):
    """
    Convert the image into patches and then project them into a vector space.
    """

    def __init__(self, config):
        super().__init__()
        self.image_size = config["image_size"]
        self.patch_size = config["patch_size"]
        self.num_channels = config["num_channels"]
        self.embed_dim = config["embed_dim"]
        self.drop = nn.Dropout(config["dropout"])
        # Calculate the number of patches from the image size and patch size
        self.num_patches = (self.image_size // self.patch_size) ** 2
        # --- CLS Token: Standard Parameter ---
        self.cls_token_param = nn.Parameter(torch.zeros(1, 1, self.embed_dim))
        # --- CLS Token: Quantizer Wrapper ---
        self.cls_token_quant = QuantIdentity(
            act_quant=Int8ActPerTensorFixedPoint,return_quant_tensor=True)
        
        # --- Positional Embedding: Standard Parameter ---
        self.pos_embed_param = nn.Parameter(torch.zeros(1, 1 + self.num_patches, self.embed_dim))
        # --- Positional Embedding: Quantizer Wrapper ---
        self.pos_embed_quant = QuantIdentity(
            act_quant=Int8ActPerTensorFixedPoint,return_quant_tensor=True)
        # Create a projection layer to convert the image into patches
        # The layer projects each patch into a vector of size hidden_size
        self.proj = QuantConv2d(
            self.num_channels, 
            self.embed_dim, 
            kernel_size=self.patch_size, 
            stride=self.patch_size,
            bias=True,
            weight_quant_type=QuantType.INT, 
            weight_bit_width=8,
            weight_restrict_scaling_type=RestrictValueType.POWER_OF_TWO,
            weight_scaling_impl_type=ScalingImplType.CONST,
            weight_scaling_const= 2**(I-1),
            bias_quant=Int8BiasPerTensorFixedPointInternalScaling
        )

    def forward(self, x):
        B = x.shape[0]

        # Project image patches: [B, C, H, W] -> [B, embed_dim, H/P, W/P]
        x = self.proj(x)                  # [B, E, H', W']
        x = x.flatten(2).transpose(1, 2)  # [B, N, E]

        # Expand CLS token to batch size and prepend to patch tokens
        cls_token = self.cls_token_param.expand(B, -1, -1)
        cls_token = self.cls_token_quant(cls_token)  # [B, 1, E]
        x = torch.cat((cls_token, x), dim=1)          # [B, N+1, E]


        # Add positional embedding
        x = x + self.pos_embed_quant(self.pos_embed_param)


        return x

In [9]:
class QuantMLP(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.fc1 = QuantLinear(
            config["embed_dim"], 
            config["mlp_size"],
            bias=True,
            weight_quant_type=QuantType.INT, 
            weight_bit_width=8,
            weight_restrict_scaling_type=RestrictValueType.POWER_OF_TWO,
            weight_scaling_impl_type=ScalingImplType.CONST,
            weight_scaling_const= 2**(I-1),
            bias_quant=Int8BiasPerTensorFixedPointInternalScaling
        )
        self.act = QuantReLU(
            quant_type=QuantType.INT,
            bit_width=W,
            restrict_scaling_type=RestrictValueType.POWER_OF_TWO,
            scaling_impl_type=ScalingImplType.CONST,
            scaling_init= 2.0**(-Fr),
        )
        self.drop = nn.Dropout(config["dropout"])
        self.fc2 = QuantLinear(
            config["mlp_size"], 
            config["embed_dim"],
            bias=True,
            weight_quant_type=QuantType.INT, 
            weight_bit_width=8,
            weight_restrict_scaling_type=RestrictValueType.POWER_OF_TWO,
            weight_scaling_impl_type=ScalingImplType.CONST,
            weight_scaling_const= 2**(I-1),
            bias_quant=Int8BiasPerTensorFixedPointInternalScaling
        )

    def forward(self,x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)

        return x

In [10]:
from brevitas.core.quant import QuantType
from brevitas.core.restrict_val import RestrictValueType
from brevitas.core.scaling import ScalingImplType


import torch
import torch.nn as nn
import torch.nn.functional as F


class QuantMultiHeadSelfAttentionLayer(nn.Module):

    def __init__(self,config):
        super().__init__()

        self.embed_dim = config["embed_dim"]
        self.num_heads = config["num_heads"]
        assert self.embed_dim % self.num_heads == 0

        self.head_dim = self.embed_dim // self.num_heads

        self.delta_tracking = True

        
        
        """
        self.q_proj = QuantLinear(
            self.embed_dim, 
            self.embed_dim,
            bias=True,
            bias_quant=Int8BiasPerTensorFixedPointInternalScaling,
            weight_quant=Int8WeightPerTensorFixedPoint,
            input_quant=Int8ActPerTensorFixedPoint)
        
                                     weight_quant_type=QuantType.INT, 
                                     weight_bit_width=8,
                                     weight_restrict_scaling_type=RestrictValueType.POWER_OF_TWO,
                                     weight_scaling_impl_type=ScalingImplType.CONST,
                                     weight_scaling_const=1.0
        """
        self.q_proj = QuantLinear(
            self.embed_dim, 
            self.embed_dim,
            bias=True,
            weight_quant_type=QuantType.INT, 
            weight_bit_width=8,
            weight_restrict_scaling_type=RestrictValueType.POWER_OF_TWO,
            weight_scaling_impl_type=ScalingImplType.CONST,
            weight_scaling_const= 2**(I-1),
            bias_quant=Int8BiasPerTensorFixedPointInternalScaling        
            )

        self.k_proj = QuantLinear(
            self.embed_dim, 
            self.embed_dim,
            bias=True,
            weight_quant_type=QuantType.INT, 
            weight_bit_width=8,
            weight_restrict_scaling_type=RestrictValueType.POWER_OF_TWO,
            weight_scaling_impl_type=ScalingImplType.CONST,
            weight_scaling_const= 2**(I-1),
            bias_quant=Int8BiasPerTensorFixedPointInternalScaling
        )


        self.v_proj = QuantLinear(
            self.embed_dim, 
            self.embed_dim,
            bias=True,
            weight_quant_type=QuantType.INT, 
            weight_bit_width=8,
            weight_restrict_scaling_type=RestrictValueType.POWER_OF_TWO,
            weight_scaling_impl_type=ScalingImplType.CONST,
            weight_scaling_const= 2**(I-1),
            bias_quant=Int8BiasPerTensorFixedPointInternalScaling
        )

        self.drop = nn.Dropout(config["dropout"])

        self.out_proj = QuantLinear(
            self.embed_dim, 
            self.embed_dim,
            bias=True,
            weight_quant_type=QuantType.INT, 
            weight_bit_width=8,
            weight_restrict_scaling_type=RestrictValueType.POWER_OF_TWO,
            weight_scaling_impl_type=ScalingImplType.CONST,
            weight_scaling_const= 2**(I-1),
            bias_quant=Int8BiasPerTensorFixedPointInternalScaling
            
        )

        
        self.logit_quant = QuantIdentity(
            quant_type=QuantType.INT,
            bit_width=32,
            restrict_scaling_type=RestrictValueType.POWER_OF_TWO,
            scaling_impl_type=ScalingImplType.CONST,
            scaling_init=2.0 ** (-Fr),  
        )


    def forward(self,x):
        #In input we have [B,N,E] -> [B,N,H,E/H] -> [B,H,N,E/H]
        #view returns a reshaped view of the same data
        B,N,D = x.shape
        q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)  # (B, H, N, head_dim)
        k = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)  # (B, H, N, head_dim)
        v = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)  # (B, H, N, head_dim)

        #After the Linear layer: Q[B,H,N,E'] * Kt[B,H,E',N]
        a = q @ k.transpose(-2,-1)

        scale = math.sqrt(self.head_dim)
        #a = F.softmax(a / scale, dim=-1)
        
        a = a / scale
        #print("Matrix A after Scale before quant:", a)
        #a = self.logit_quant(a)
        
        a = F.softmax(a,dim=-1)
        
        
        a = self.drop(a)

        #view() can only reinterpret a contiguous tensor; if it’s not, you must first make it contiguous
        #Now we matmul A[B,H,N,N] x V[B,H,N,E'] = [B,H,N,E'] -
        p = (a @ v).transpose(1,2).contiguous().view(B,N,D)
        return self.out_proj(p)
        



In [11]:
class QuantBlockBatch(nn.Module):
    def __init__(self, config):
        super().__init__()
        D = config["embed_dim"]
        self.mha = QuantMultiHeadSelfAttentionLayer(config)
        self.mlp = QuantMLP(config)
        self.bn1 = nn.BatchNorm1d(D)
        self.bn2 = nn.BatchNorm1d(D)

    def _bn(self, bn, x):              # x: [B, N, D]
        x = x.transpose(1, 2)          # [B, D, N]
        x = bn(x)                      # BN over features
        return x.transpose(1, 2)       # back to [B, N, D]

    def forward(self, x):
        y = self.mha(self._bn(self.bn1, x))
        x = x + y
        z = self.mlp(self._bn(self.bn2, x))
        return x + z

In [12]:
class ViT(nn.Module):
    def __init__(self,config):
        super().__init__()
        embed_dim = config["embed_dim"]
        depth = config["depth"]
        num_classes = config["num_classes"]

        self.patch = QuantPatchEmbeddings(config)
        #self.norm = nn.LayerNorm(embed_dim)
        self.norm = nn.BatchNorm1d(embed_dim)
        self.blocks = nn.ModuleList([
            #QuantBlock(config)
            QuantBlockBatch(config)
            for _ in range(depth)
        ])

        self.head = QuantLinear(
            embed_dim, 
            num_classes,
            bias=True,
            weight_quant_type=QuantType.INT, 
            weight_bit_width=8,
            weight_restrict_scaling_type=RestrictValueType.POWER_OF_TWO,
            weight_scaling_impl_type=ScalingImplType.CONST,
            weight_scaling_const= 2**(I-1),
            bias_quant=Int8BiasPerTensorFixedPointInternalScaling
        )
        
    def _bn(self, bn, x):              # x: [B, N, D]
        x = x.transpose(1, 2)          # [B, D, N]
        x = bn(x)                      # BN over features, if not transpose we bn over tokens
        return x.transpose(1, 2) 

    def forward(self,x):

        x = self.patch(x)  # (B, N, D)
        for block in self.blocks:
            x = block(x)
        x = self._bn(self.norm,x)
        cls_token = x[:, 0]  # Use CLS token
        return self.head(cls_token)



        

In [14]:

model = ViT(config).to(device)

EPOCHS = config["epochs"]

EPOCHS = 10

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)

train_ = True

if train_:

      for epoch in range(EPOCHS):
            train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
            test_loss, test_acc = test(model, test_loader, criterion, device)

            print(f"[Epoch {epoch+1}/{EPOCHS}] "
                  f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
                  f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

      model.to("cpu")
      torch.save(model.state_dict(), "ViT_Quant.pth")
else:
      model.load_state_dict(torch.load("ViT_Quant.pth"))
      model.eval()

/home/fede/miniconda3/envs/ptorch/lib/python3.9/site-packages/torch/_tensor.py:1413: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at /opt/conda/conda-bld/pytorch_1720538643151/work/c10/core/TensorImpl.h:1925.)
  return super().rename(names)


[Epoch 1/10] Train Loss: 0.9035, Train Acc: 0.6882 | Test Loss: 0.5397, Test Acc: 0.8136
[Epoch 2/10] Train Loss: 0.5333, Train Acc: 0.8224 | Test Loss: 0.4177, Test Acc: 0.8592
[Epoch 3/10] Train Loss: 0.4680, Train Acc: 0.8438 | Test Loss: 0.3969, Test Acc: 0.8652
[Epoch 4/10] Train Loss: 0.4386, Train Acc: 0.8548 | Test Loss: 0.3731, Test Acc: 0.8775
[Epoch 5/10] Train Loss: 0.4132, Train Acc: 0.8636 | Test Loss: 0.3510, Test Acc: 0.8843
[Epoch 6/10] Train Loss: 0.3943, Train Acc: 0.8700 | Test Loss: 0.3593, Test Acc: 0.8831
[Epoch 7/10] Train Loss: 0.3822, Train Acc: 0.8754 | Test Loss: 0.3450, Test Acc: 0.8895
[Epoch 8/10] Train Loss: 0.3728, Train Acc: 0.8763 | Test Loss: 0.3455, Test Acc: 0.8863
[Epoch 9/10] Train Loss: 0.3659, Train Acc: 0.8790 | Test Loss: 0.3412, Test Acc: 0.8896
[Epoch 10/10] Train Loss: 0.3549, Train Acc: 0.8818 | Test Loss: 0.3065, Test Acc: 0.8981


In [ ]:
import torch

activations = {}

def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook

targets = ["blocks.0.mha.softmax","patch", "blocks.0.mha","blocks.0.mha.v_proj","blocks.0.mha.q_proj","blocks.0.mha.k_proj", "blocks.0.mlp", "norm", "blocks.0.bn1", "blocks.0.bn2"]
activations = {}

for name, module in model.named_modules():
    if name in targets:
        module.register_forward_hook(get_activation(name))



In [ ]:
model.to("cpu")



#def test(model, dataloader, criterion, device):
model.eval()

with torch.no_grad():
    images, labels = next(iter(test_loader))   # images: [B, 1, 28, 28]
    # Optionally pick just the first sample:
    img = images[0].unsqueeze(0)              # [1, 1, 28, 28]
    lbl = labels[0].unsqueeze(0)              # [1]

    outputs = model(img)
    loss = criterion(outputs, lbl)

    print(f"Image, Label: {lbl}, Output: {outputs}, Loss: {loss.item()}")

N = 50
H = 8
H_dim = 16

print("img shape:", img.shape)

outputs_cpu = outputs.cpu().numpy()
img_cpu = img.permute(0, 2, 3, 1).cpu().numpy().flatten()
lbl_cpu = lbl.cpu().numpy()
patch_o = activations["patch"].cpu().numpy().flatten()
patch_o_tp = activations["patch"].cpu().numpy().transpose(0, 2, 1).flatten()
norm_patch_o = activations["blocks.0.bn1"].cpu().numpy().transpose(0, 2, 1).flatten()  
norm_patch_2_o = activations["blocks.0.bn2"].cpu().numpy().transpose(0, 2, 1).flatten()
mha_o = activations["blocks.0.mha"].cpu().numpy().flatten()
mlp_o = activations["blocks.0.mlp"].cpu().numpy().flatten()
q_linear = activations["blocks.0.mha.q_proj"].cpu().view(1, N, H, H_dim).transpose(1, 2)#.contiguous().flatten()
v_linear = activations["blocks.0.mha.v_proj"].cpu().view(1, N, H, H_dim).transpose(1, 2)#.contiguous().flatten()
k_linear = activations["blocks.0.mha.k_proj"].cpu().view(1, N, H, H_dim).transpose(1, 2)#.contiguous()#.flatten()

softmax_module = activations["blocks.0.mha.softmax"].cpu().numpy().flatten()
q_linear_np = q_linear.detach().cpu().numpy()
k_linear_tp = k_linear.transpose(-2,-1)


print("k linear shape:", k_linear.shape)
print("k linear tp shape:", k_linear_tp.shape)



k_linear_np = k_linear.detach().cpu().numpy()
k_linear_np_tp = k_linear_tp.detach().cpu().numpy()
v_linear_np = v_linear.detach().cpu().numpy()

matrix_a = q_linear @ k_linear_tp
matrix_a = matrix_a / 4.0
matrix_a = F.softmax(matrix_a, dim=-1)

matrix_mha = matrix_a @ v_linear

print("Matrix MHA:", matrix_mha)

np.savetxt("mnist_output.txt", outputs_cpu.reshape(-1, 1), fmt="%.16f")
np.savetxt("mnist_input.txt", img_cpu, fmt="%.16f")
np.savetxt("mnist_label.txt", lbl_cpu, fmt="%.16f")
np.savetxt("patch_output.txt", patch_o, fmt="%.16f")
np.savetxt("patch_output_tp.txt", patch_o_tp, fmt="%.16f")
np.savetxt("norm_0.txt", norm_patch_o, fmt="%.16f")
np.savetxt("norm_1.txt", norm_patch_2_o, fmt="%.16f")
np.savetxt("mha_0.txt", mha_o, fmt="%.16f")
np.savetxt("mlp_0.txt", mlp_o, fmt="%.16f")
np.savetxt("q.txt", q_linear_np.flatten(), fmt="%.16f")
np.savetxt("v.txt", v_linear_np.flatten(), fmt="%.16f")
np.savetxt("softmax_logits.txt", softmax_module, fmt="%.16f")
np.savetxt("k.txt", k_linear_np.flatten(), fmt="%.16f")
np.savetxt("k_tp.txt", k_linear_np_tp.flatten(), fmt="%.16f")
np.savetxt("a.txt", matrix_a.flatten(), fmt="%.16f")




With the following code we can save the positional encoding and the cls token parameters

In [ ]:
import math
from math import log2, ceil

model.to('cpu')
model_dict = model.state_dict()
printing = False

if printing == False:
    
    
    # 1) Grab the fractional bits f from the quantizer
    f = abs(int(model_dict['patch.cls_token_quant.act_quant.fused_activation_quant_proxy'
                    '.tensor_quant.scaling_impl.value']) )

    print("fractional bits f:", f)

    f = 16
    # 2) Fetch the FP32 cls token
    cls = model_dict['patch.cls_token_param'].squeeze().numpy()  # shape [D]
    pos_enc = model_dict['patch.pos_embed_param'].squeeze().numpy()  # shape [N+1,D]

    # 3) Emulate Brevitas fixed-point int8 quant/dequant
    q_int = np.round(cls * (2 ** f))  # integer code
    pos_int = np.round(pos_enc * (2 ** f))   # integer code
    cls_q = q_int / (2 ** f)                               # dequantized value
    pos_q = pos_int / (2 ** f)                             # dequantized value

    # 4) Compare
    print("float     :", cls)
    print("quantized :", cls_q)
    print("max |err| :", np.max(np.abs(cls - cls_q)))
    print("cls shape:", cls.shape)

    with open('/home/fede/PhD/ViT_Brevitas/notebook/CLS_PARAM.h', "w") as f:
        f.write("// Auto-generated. int8 weights; scale = 2^-f_w\n")
        f.write(f"#pragma once\n#include <ap_int.h>\n#include <stdint.h>\n\n")

        

        f.write(f"#define CLS_PARAM {{  ")
        vals = ", ".join(str(float(v)) for v in cls_q)
        f.write(vals + "}")
    
    with open('/home/fede/PhD/ViT_Brevitas/notebook/POS_ENC_PARAM.h', "w") as f:
        f.write("// Auto-generated. int8 weights; scale = 2^-f_w\n")
        f.write(f"#pragma once\n#include <ap_int.h>\n#include <stdint.h>\n\n")


        f.write(f"#define POS_ENC_PARAM  {{  ")
        for row in pos_q:
            vals = ", ".join(str(float(v)) for v in row)
            f.write("{" + vals + "},")
        f.write("}")


if printing == True:
    for key in model_dict.keys():
        # Filter for potential CLS token keys
        if 'cls' in key or 'token' in key:
            print(f"Potential CLS Token Key: {key}")
        else:
            # Print a few others to see the naming convention (e.g., 'patch_embed.proj.weight')
            print(key)


In [ ]:
import numpy as np, os

def export_linear_int8(layer, name, out_dir, f_w: int):
    # layer.weight: [out, in], layer.bias: [out] (optional)
    W = layer.weight.detach().cpu().numpy() if layer.weight is not None else None       # float
    B = layer.bias.detach().cpu().numpy() if layer.bias is not None else None
    scale = layer.quant_weight()
    # If you already have int8 codes from Brevitas, use those directly.
    # Otherwise quantize: q = clip(round(W * 2^f_w), -128..127)
    qW = np.round(W * (2**f_w))/(2**f_w) if W is not None else None
    bW = np.round(B * (2**f_w))/(2**f_w) if B is not None else None

    path = os.path.join(out_dir, f"{name}.h")
    qW = qW.transpose(1,0)
    R, C = qW.shape  # rows=out, cols=in
    print(bW.shape)
    with open(path, "w") as f:
        f.write("// Auto-generated. int8 weights; scale = 2^-f_w\n")
        f.write(f"#pragma once\n#include <ap_int.h>\n#include <stdint.h>\n#include \"types.h\"\n\n")
        f.write(f"static const act_0 {name}_W[{R}][{C}] = {{\n")
        for r in range(R):
            row = ", ".join(str(float(v)) for v in qW[r])
            f.write(f"  {{{row}}}")
            f.write(",\n" if r < R-1 else "\n")
        f.write("};\n")

        f.write(f"static const act_0 {name}_B[{C}] = {{\n  ")
        vals = ", ".join(str(float(v)) for v in bW)
        f.write(vals + "\n};\n")

        

    print("wrote", path)


export_linear_int8(model.blocks[0].mlp.fc1, "MLP1", "/home/fede/PhD/Weight_", f_w=Fr)
export_linear_int8(model.blocks[0].mlp.fc2, "MLP2", "/home/fede/PhD/Weight_", f_w=Fr)
export_linear_int8(model.blocks[0].mha.q_proj, "Q_", "/home/fede/PhD/Weight_", f_w=Fr)
export_linear_int8(model.blocks[0].mha.k_proj, "K_", "/home/fede/PhD/Weight_", f_w=Fr)
export_linear_int8(model.blocks[0].mha.v_proj, "V_", "/home/fede/PhD/Weight_", f_w=Fr)
export_linear_int8(model.blocks[0].mha.out_proj, "OUT_", "/home/fede/PhD/Weight_", f_w=Fr)
export_linear_int8(model.head, "HEAD_", "/home/fede/PhD/Weight_", f_w=Fr)




In [ ]:
import os
import numpy as np

def export_conv_int8(layer, name, out_dir, f_w: int):
    """
    Export a Brevitas Conv2d layer as C macros:
      #define WEIGHT_<name>  {{{{...k elems...}, ...}, ...}}
      #define BIAS_<name>    {...}

    Layout:
      PyTorch/Brevitas weights: [OC, IC, KH, KW]
      Exported macro order:     [IC, KH, KW, OC]  (innermost = OC = kern_s_k_<name>)
    Quantization:
      q = clip(round(x * 2^f_w), -128..127) / 2^f_w
    """
    os.makedirs(out_dir, exist_ok=True)

    # --- 1) Pull weights/biases from the Brevitas layer ---
    # If you already store quantized codes elsewhere, feel free to replace W/B with those.
    W = layer.weight.detach().cpu().numpy() if getattr(layer, "weight", None) is not None else None
    B = layer.bias.detach().cpu().numpy()   if getattr(layer, "bias",   None) is not None else None

    if W is None:
        raise ValueError(f"Layer {name} has no weights.")
    
    print("Original W shape:", W.shape)

    # --- 2) Quantize to int8 grid with f_w fractional bits (same policy as your linear) ---
    scale = float(2 ** f_w)
    qW = np.round(W * scale) / scale
    if B is not None:
        bW = np.round(B * scale)/ scale
    else:
        bW = None

    # --- 3) Reorder to [IC, KH, KW, OC] so inner-most dimension is OC (kern_s_k_<name>) ---
    # Original: [OC, IC, KH, KW]
    qW_re = np.transpose(qW, (1, 2, 3, 0))  # -> [IC, KH, KW, OC]

    IC, KH, KW, OC = qW_re.shape

    # --- 4) Pretty-printers for nested C-like brace formatting ---
    def fmt_num(x: float) -> str:
        s = f"{x:.6g}"
        if s == "-0":
            s = "-0.0"
        elif s == "0":
            s = "0.0"
        return s

    def format_4d_ic_h_w_oc(arr: np.ndarray) -> str:
        # arr shape: [IC, KH, KW, OC]
        # Emit as {{{{oc-list}, {oc-list}, ...}, ...}, ...}
        out_ic = []
        for ic in range(arr.shape[0]):
            out_h = []
            for h in range(arr.shape[1]):
                out_w = []
                for w in range(arr.shape[2]):
                    oc_list = ", ".join(fmt_num(v) for v in arr[ic, h, w, :].tolist())
                    out_w.append("{" + oc_list + "}")
                out_h.append("{" + ", ".join(out_w) + "}")
            out_ic.append("{" + ", ".join(out_h) + "}")
        return "{" + ", ".join(out_ic) + "}"

    def format_1d(arr: np.ndarray) -> str:
        return "{" + ", ".join(fmt_num(x) for x in arr.tolist()) + "}"

    weight_macro  = f"#define WEIGHT_{name}   " + format_4d_ic_h_w_oc(qW_re)
    bias_macro    = f"#define BIAS_{name}   "  + (format_1d(bW) if bW is not None else "{}")

    # --- Write header file ---
    path = os.path.join(out_dir, f"{name}.h")
    with open(path, "w") as f:
        f.write("// Auto-generated from Brevitas Conv2d. int8 grid; scale = 2^-f_w\n")
        f.write("#pragma once\n#include <stdint.h>\n\n")
        f.write(weight_macro + "\n\n")
        f.write(bias_macro   + "\n")

    print("wrote", path)
    return path

export_conv_int8(model.patch.proj, "PATCH", "/home/fede/PhD/Weight_", f_w=16)

In [ ]:
model.eval()

batch_input = activations["patch"].cpu().numpy()
print("Batch input shape:", batch_input.shape)  # Should be [1, N, D]

model.blocks[0].bn1.eval()

"""
def _bn(self, bn, x):              # x: [B, N, D]
    x = x.transpose(1, 2)          # [B, D, N]
    x = bn(x)                      # BN over features
    return x.transpose(1, 2) 
"""

with torch.no_grad():
    batch_tensor = torch.tensor(batch_input.transpose(0, 2, 1), dtype=torch.float32)  # [B, D, N]
    norm_output = model.blocks[0].bn1(batch_tensor)  # [B, N, D]

    bn = model.blocks[0].bn1
    W  = bn.weight.detach().cpu().numpy()       # gamma
    B  = bn.bias.detach().cpu().numpy()         # beta
    RM = bn.running_mean.detach().cpu().numpy()
    RV = bn.running_var.detach().cpu().numpy()
    eps = bn.eps

    inv_std = 1.0 / np.sqrt(RV + eps)

    c   = 0         # channel index
    inp = batch_tensor[0][0][0]   # your input

    manual_fp = (inp - RM[c]) * inv_std[c] * W[c] + B[c]
    #Debugs
    print("manual_fp:", manual_fp)
    print("torch_bn:", norm_output[0, c, 0].item())

    def quantize_fixed32(value, f=16):
        scale = 2**f
        max_int = 2**31 - 1
        min_int = -2**31
        return np.round(value * scale) / scale
    qW = quantize_fixed32(W, f=16)
    qB = quantize_fixed32(B, f=16)
    qRM = quantize_fixed32(RM, f=16)
    qRV = quantize_fixed32(RV, f=16)

    

    inv_std = 1.0 / np.sqrt(RV + eps)
    qRunningVar = 1.0 / np.sqrt(qRV + eps)
    qInvStd = quantize_fixed32(inv_std, f=16)

    manual_fp_q = (inp - qRM[c]) * qInvStd[c] * qW[c] + qB[c]

    print("Input value:", batch_tensor[0, c, 0].item())
    print("manual_fp_q:", manual_fp_q)
    print("qRM:", qRM[c])
    print("qInvStd:", qInvStd[c])
    print("qRV:", qRV[c])
    print("qW:", qW[c])
    print("qB:", qB[c])

In [ ]:
"""
norm.weight
norm.bias
norm.running_mean
norm.running_var

"""
import os
import numpy as np
from math import sqrt

def export_batch_int8(layer, name, out_dir, f_w: int):
    # layer.weight: [out, in], layer.bias: [out] (optional)

    W = layer.weight.detach().cpu().numpy() if layer.weight is not None else None       # float
    B = layer.bias.detach().cpu().numpy() if layer.bias is not None else None
    RM = layer.running_mean.detach().cpu().numpy() if layer.weight is not None else None   
    RV = layer.running_var.detach().cpu().numpy() if layer.weight is not None else None 
    
    # If you already have int8 codes from Brevitas, use those directly.
    # Otherwise quantize: q = clip(round(W * 2^f_w), -128..127)
    """
    qW = np.clip(np.round(W * (2**f_w)), -2**16, 2**16 - 1)/(2**f_w) if W is not None else None
    bW = np.clip(np.round(B * (2**f_w)), -2**16, 2**16 - 1)/(2**f_w) if B is not None else None
    qRM = np.clip(np.round(RM * (2**f_w)), -2**16, 2**16 - 1)/(2**f_w) if W is not None else None
    qRV = np.clip(np.round(RV * (2**f_w)), -2**16, 2**16 - 1)/(2**f_w) if B is not None else None
    """
    def quantize_fixed32(value, f=16):
        scale = 2**f
        max_int = 2**31 - 1
        min_int = -2**31
        return np.round(value * scale)/ scale
    
    qW = quantize_fixed32(W, f=f_w)
    bW = quantize_fixed32(B, f=f_w)
    qRM = quantize_fixed32(RM, f=f_w)
    qRV = quantize_fixed32(RV, f=f_w)

    eps = layer.eps

    inv_std = 1.0 / np.sqrt(RV + eps)
    qRunningVar = 1.0 / np.sqrt(qRV + eps)
    qInvStd = quantize_fixed32(inv_std, f=f_w)

    """
    print("Example BatchNorm parameters:")
    print("qInvStd:", qInvStd)
    print("W:", W)
    print("B:", B)
    print("Running Mean:", RM)
    print("Running Var:", RV)
    """


    path = os.path.join(out_dir, f"{name}.h")
    R = qW.shape  # rows=out, cols=in
    with open(path, "w") as f:
        f.write("// Auto-generated. int8 weights; scale = 2^-f_w\n")
        f.write(f"#pragma once\n#include <ap_int.h>\n#include <stdint.h>\n\n")
        
        f.write(f"#define {name}_W {{  ")
        vals = ", ".join(str(float(v)) for v in qW)
        f.write(vals + "} \n")

        f.write(f"#define {name}_B {{  ")
        vals = ", ".join(str(float(v)) for v in bW)
        f.write(vals + "} \n")

        f.write(f"#define {name}_Running_Mean {{  ")
        vals = ", ".join(str(float(v)) for v in qRM)
        f.write(vals + "} \n")

        f.write(f"#define {name}_Running_Var {{  ")
        vals = ", ".join(str(float(v)) for v in qInvStd)
        f.write(vals + "} \n")

    print("wrote", path)


export_batch_int8(model.norm, "NORM", "/home/fede/PhD/Weight_", f_w=16)
export_batch_int8(model.blocks[0].bn1, "NORM_1", "/home/fede/PhD/Weight_", f_w=16)
export_batch_int8(model.blocks[0].bn2, "NORM_2", "/home/fede/PhD/Weight_", f_w=16)


In [15]:
from brevitas.export import export_qonnx
from brevitas.quant import Int8ActPerTensorFloat, Int16Bias

float_inp = torch.randn(1, 1,28,28)

model.to("cpu")
model.eval()

output_path = './qvit_onnx.onnx'
exported_model = export_qonnx(model, input_t=float_inp, export_path=output_path)

This is a default, simple LUT for exp. I would like to move from a LUT implementation to an approximated version like it's done in the ITA Paper (https://arxiv.org/abs/2307.03493)

In [ ]:
# --- CONFIG ---
N_ENTRIES = 256          # number of LUT entries
Z_MIN     = -8.0         # minimum z
Z_MAX     = 0.0          # maximum z
CPP_TYPE  = "AccType"    # C++ type used in your LUT (e.g. AccType)

# --- COMPUTE ---
step = (Z_MAX - Z_MIN) / (N_ENTRIES - 1)

values = []
for i in range(N_ENTRIES):
    z = Z_MIN + i * step
    e = math.exp(z)
    values.append(e)

# --- PRINT C++ INITIALIZER ---
print(f"// exp(z) LUT, z in [{Z_MIN}, {Z_MAX}], {N_ENTRIES} entries")
print(f"static const {CPP_TYPE} LUT[{N_ENTRIES}] = {{")

line = "    "
for i, v in enumerate(values):
    line += f"({CPP_TYPE}){v:.10e}, "
    # break lines every 4 or 8 values for readability
    if (i + 1) % 4 == 0:
        print(line)
        line = "    "

if line.strip():
    print(line)

print("};")
